# Week 8: SGP4 propagation in Python with skyfield

**Track:** Orbital Analyst (Intermediate)  
**Full primer + quiz:** [https://launchdetect.com/academy/week/8/](https://launchdetect.com/academy/week/8/)  
**Track index:** [https://launchdetect.com/academy/orbital-analyst/](https://launchdetect.com/academy/orbital-analyst/)

---

_skyfield is Python's premier orbit-propagation library. SGP4 is the workhorse algorithm. This week you'll propagate the ISS, generate ground tracks, and produce a publishable GeoJSON._


## Why this week matters

A TLE describes the orbit in the abstract. SGP4 propagation tells you *where the satellite is right now* — and it's the answer to almost every operational question in space situational awareness: when does the next pass occur, what's the satellite overflying, can the ground station see it, what's the visibility window for imaging.

**Why SGP4 specifically?** Because it's the public-standard propagator. Every amateur tracker, every public-data feed, every Web service that takes a TLE and returns a position uses SGP4 internally. It's not the most accurate propagator available — high-fidelity numerical propagators give cm-precision for geodetic satellites — but it's accurate enough for almost everything in the civilian space domain (sub-km for fresh TLEs), and it's universally interoperable.

`skyfield` is Python's reference implementation. It wraps the canonical SGP4 C library with a clean Python API and adds proper time-system handling (UTC vs UT1 vs TT vs TDB), Earth orientation, and astronomical math you'll need in later weeks.


## Learning objectives

By the end of this lab you will be able to:

- Use skyfield to load a TLE and propagate it
- Compute sub-satellite point (lat/lon directly below) at any UTC time
- Generate a 24-hour ground track as a GeoJSON LineString
- Handle the timing subtlety of converting between UTC, TT, and TLE epoch


## Setup — and why these dependencies

`skyfield` is the workhorse. On first use it downloads ~10 MB of IERS Earth-orientation tables (the corrections needed for sub-arc-second accuracy). Cache them; don't re-download every run.


In [ ]:
# Install required packages
!pip install -q leafmap[common] skyfield geopandas shapely


## The approach (and why)

Propagate the ISS for 24 hours at 1-minute resolution (1,440 points) using SGP4, compute the sub-satellite point at each step, and emit a GeoJSON LineString.

**Why 1-minute resolution?** Smooth enough for visualization (the orbit changes ~4° per minute) and cheap to compute (~50 ms total for 1440 points). For operational tracking apps, 1 Hz is realistic; for visualization, 1-minute is plenty.

**Why split on antimeridian crossings?** Because a polyline that goes from +179° lon to -179° lon will draw a line across the entire map (almost all the way around) instead of jumping the dateline. Splitting at the discontinuity is the standard fix.

**Why GeoJSON LineString?** Universal interchange format. QGIS opens it, MapLibre/Leaflet ingest it, PostGIS imports it. No proprietary format wins anywhere.


In [ ]:
# Week 8: 24-hour ISS ground track via skyfield SGP4.
import leafmap.foliumap as leafmap
from skyfield.api import EarthSatellite, load, wgs84

tle1 = "1 25544U 98067A   24130.50145833  .00018539  00000-0  33188-3 0  9994"
tle2 = "2 25544  51.6406 348.5395 0006703 117.9568 358.1729 15.50289267449420"
ts = load.timescale()
iss = EarthSatellite(tle1, tle2, "ISS", ts)

# 24 hours at 1-minute resolution
t0 = ts.now()
times = [ts.tt_jd(t0.tt + i/1440) for i in range(0, 1440)]
points = [(wgs84.subpoint(iss.at(t)).longitude.degrees,
           wgs84.subpoint(iss.at(t)).latitude.degrees) for t in times]

# Split on antimeridian crossings (so the line doesn't draw across the map)
segments = [[points[0]]]
for p in points[1:]:
    prev_lon = segments[-1][-1][0]
    if abs(p[0] - prev_lon) > 180:
        segments.append([p])
    else:
        segments[-1].append(p)

fc = {"type": "FeatureCollection", "features": [
    {"type": "Feature", "geometry": {"type": "LineString", "coordinates": seg},
     "properties": {"name": f"ISS ground track segment {i+1}"}}
    for i, seg in enumerate(segments) if len(seg) > 1
]}

m = leafmap.Map(center=[0, 0], zoom=2)
m.add_geojson(fc, style={"color": "#4a9eff", "weight": 2})
m

# TODO: refresh the TLE from CelesTrak before propagating, and add altitude as
# a popup on each segment.


## What just happened — and why it works

`skyfield`'s `EarthSatellite(line1, line2, name, ts).at(t).subpoint()` returns a WGS84 lat/lon/altitude tuple for time `t`. The conversion from the propagator's internal ECEF/inertial frame to lat/lon happens inside the library — you get a ready-to-use ground point.

The antimeridian-splitting loop walks the ground track and starts a new line segment whenever the longitude jumps by more than 180° between consecutive points. That heuristic detects the dateline crossing without explicit geometry math.

The result is a multi-segment LineString: each segment is a continuous orbital pass from one antimeridian crossing to the next. For the ISS, expect ~16 segments over 24 hours (16 orbits / day, but some cross the antimeridian on ascending and descending separately).


## Common gotchas

- **Don't use `ts.now()` for reproducible runs.** Use `ts.utc(2026, 5, 11)` or similar so results are deterministic.
- **The TLE epoch matters more than the propagation start time.** Propagating 30 days past the epoch introduces km-scale error for LEO objects.
- **Sub-satellite altitude oscillates ~10 km over an orbit** due to eccentricity. If you're color-coding altitude, you'll see a slow up-down pattern, not a constant.


## Self-check

Verify your solution against these checks before considering the lab complete:

- [ ] Output type matches the expected format (GeoJSON / PNG / table / etc.).
- [ ] No exceptions raised on a clean run.
- [ ] Result is visually plausible — open the map cell, scan the values, sanity-check magnitudes.
- [ ] Quiz on the [week page](https://launchdetect.com/academy/week/8/) — try answering before checking the key.

---

Found a bug or want to contribute an improvement? Open an issue or PR at [github.com/launchdetect/academy-labs](https://github.com/launchdetect/academy-labs).
